In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: c:\Users\Mehak\OneDrive\MEHAK\Desktop\RetailPulse
src exists: True


In [5]:
from src.data.loader import load_raw_data

df_raw = load_raw_data()

print("Dataset shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())

print("\nRecords by source sheet:")
print(df_raw["SourceSheet"].value_counts())

print("\nFirst five rows:")
display(df_raw.head())

Dataset shape: (1067371, 9)

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'SourceSheet']

Records by source sheet:
SourceSheet
2010-2011    541910
2009-2010    525461
Name: count, dtype: int64

First five rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010


In [6]:
from src.data.validation import validate_schema

validate_schema(df_raw)

RETAILPULSE - SCHEMA VALIDATION

All required columns are present.
No unexpected columns found.

DATA TYPE CHECK
Quantity: Numeric - OK
Price: Numeric - OK
InvoiceDate: Datetime - OK

MISSING VALUE CHECK
Invoice: 0 missing values (0.00%)
StockCode: 0 missing values (0.00%)
Description: 4,382 missing values (0.41%)
Quantity: 0 missing values (0.00%)
InvoiceDate: 0 missing values (0.00%)
Price: 0 missing values (0.00%)
Customer ID: 243,007 missing values (22.77%)
Country: 0 missing values (0.00%)
SourceSheet: 0 missing values (0.00%)

DATASET SUMMARY
Rows: 1,067,371
Columns: 9

Schema validation completed.


In [7]:
from src.data.duplicates import audit_duplicates

duplicate_records = audit_duplicates(df_raw)

RETAILPULSE - DUPLICATE AUDIT

DUPLICATE SUMMARY
Total records: 1,067,371
Extra duplicate copies: 34,335
Rows belonging to duplicate groups: 67,242

DUPLICATE RECORDS BY SOURCE SHEET
SourceSheet
2009-2010    35189
2010-2011    32053
Name: count, dtype: int64

SAMPLE DUPLICATE RECORDS


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,2009-2010
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,2009-2010



Duplicate audit completed.


In [8]:
from src.data.duplicates import ORIGINAL_COLUMNS

# Group records that are identical across all original columns
duplicate_groups = (
    df_raw.groupby(ORIGINAL_COLUMNS, dropna=False)
    .agg(
        total_rows=("SourceSheet", "size"),
        source_count=("SourceSheet", "nunique"),
        source_sheets=("SourceSheet", "unique")
    )
    .reset_index()
)

# Keep only groups containing repeated records
duplicate_groups = duplicate_groups[
    duplicate_groups["total_rows"] > 1
]

# Separate groups that occur across both sheets
cross_sheet_groups = duplicate_groups[
    duplicate_groups["source_count"] > 1
]

within_sheet_groups = duplicate_groups[
    duplicate_groups["source_count"] == 1
]

print("DUPLICATE GROUP ANALYSIS")
print("-" * 45)

print("Total duplicate groups:", len(duplicate_groups))

print("\nWithin-sheet duplicate groups:")
print(len(within_sheet_groups))

print("\nCross-sheet duplicate groups:")
print(len(cross_sheet_groups))

print("\nRows in cross-sheet duplicate groups:")
print(cross_sheet_groups["total_rows"].sum())

print("\nExtra copies in cross-sheet groups:")
print(
    (
        cross_sheet_groups["total_rows"] - 1
    ).sum()
)

print("\nSample cross-sheet duplicate groups:")
display(cross_sheet_groups.head(10))

DUPLICATE GROUP ANALYSIS
---------------------------------------------
Total duplicate groups: 32907

Within-sheet duplicate groups:
10705

Cross-sheet duplicate groups:
22202

Rows in cross-sheet duplicate groups:
45046

Extra copies in cross-sheet groups:
22844

Sample cross-sheet duplicate groups:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,total_rows,source_count,source_sheets
486538,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,2,2,"[2009-2010, 2010-2011]"
486539,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,2,2,"[2009-2010, 2010-2011]"
486540,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2,2,"[2009-2010, 2010-2011]"
486541,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2,2,"[2009-2010, 2010-2011]"
486542,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2,2,"[2009-2010, 2010-2011]"
486543,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2,2,"[2009-2010, 2010-2011]"
486544,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2,2,"[2009-2010, 2010-2011]"
486545,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,2,2,"[2009-2010, 2010-2011]"
486546,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,2,2,"[2009-2010, 2010-2011]"
486547,536367,21754,HOME BUILDING BLOCK WORD,3,2010-12-01 08:34:00,5.95,13047.0,United Kingdom,2,2,"[2009-2010, 2010-2011]"


In [9]:
from src.data.duplicates import ORIGINAL_COLUMNS

# Preserve the original dataset
df_deduplicated = df_raw.copy()

# Remove exact duplicate records, keeping the first occurrence
df_deduplicated = df_deduplicated.drop_duplicates(
    subset=ORIGINAL_COLUMNS,
    keep="first"
).copy()

print("DUPLICATE REMOVAL SUMMARY")
print("-" * 40)

print("Original rows:", len(df_raw))
print("Rows after deduplication:", len(df_deduplicated))
print("Rows removed:", len(df_raw) - len(df_deduplicated))

print("\nRemaining duplicate rows:")
print(
    df_deduplicated.duplicated(
        subset=ORIGINAL_COLUMNS
    ).sum()
)

DUPLICATE REMOVAL SUMMARY
----------------------------------------
Original rows: 1067371
Rows after deduplication: 1033036
Rows removed: 34335

Remaining duplicate rows:
0


In [10]:
from src.data.preprocessing import separate_transaction_types

(
    df_purchases,
    df_cancellations,
    df_returns,
    df_zero_quantity
) = separate_transaction_types(df_deduplicated)

RETAILPULSE - TRANSACTION SEPARATION
Original records: 1,033,036

TRANSACTION CATEGORIES
Ordinary purchase events: 1,010,539
Cancellation records: 19,104
Return records: 3,393
Zero-quantity records: 0

Records accounted for: 1,033,036
Unaccounted records: 0

Transaction separation completed.


In [13]:
import importlib
import src.data.preprocessing as preprocessing

importlib.reload(preprocessing)

df_customer, df_product = preprocessing.handle_missing_values(
    df_purchases
)

RETAILPULSE - MISSING VALUE HANDLING
Purchase records: 1,010,539

Records with identified customers: 779,495
Records without Customer ID: 231,044

Records available for product analysis: 1,010,539
Records without Description: 1,642

No records permanently deleted.


In [14]:
import importlib
import src.data.preprocessing as preprocessing

importlib.reload(preprocessing)

df_valid_price, df_zero_price, df_negative_price = (
    preprocessing.validate_transaction_prices(df_product)
)

RETAILPULSE - PRICE VALIDATION
Total purchase records: 1,010,539
Positive-price records: 1,007,913
Zero-price records: 2,621
Negative-price records: 5

Price validation completed.
No records permanently deleted.


In [15]:
import importlib
import src.data.preprocessing as preprocessing

importlib.reload(preprocessing)

df_transactions = preprocessing.calculate_transaction_values(
    df_valid_price
)

display(df_transactions.head())

RETAILPULSE - TRANSACTION VALUE CALCULATION
Records processed: 1,007,913
Total recorded transaction value: GBP 20,476,260.45
Minimum transaction value: GBP 0.00
Maximum transaction value: GBP 168,469.60

Transaction value calculation completed.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,TransactionValue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-2010,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010,30.0


In [16]:
print("FINAL DATA SANITY CHECK")
print("-" * 45)

print("Rows:", len(df_transactions))

print("\nQuantity:")
print("Minimum:", df_transactions["Quantity"].min())
print("Maximum:", df_transactions["Quantity"].max())
print("Non-positive:", (df_transactions["Quantity"] <= 0).sum())

print("\nPrice:")
print("Minimum:", df_transactions["Price"].min())
print("Maximum:", df_transactions["Price"].max())
print("Non-positive:", (df_transactions["Price"] <= 0).sum())

print("\nTransaction Value:")
print("Minimum:", repr(df_transactions["TransactionValue"].min()))
print("Non-positive:",
      (df_transactions["TransactionValue"] <= 0).sum())

print("\nDate range:")
print(df_transactions["InvoiceDate"].min())
print(df_transactions["InvoiceDate"].max())

print("\nMissing values:")
print(df_transactions.isnull().sum())

FINAL DATA SANITY CHECK
---------------------------------------------
Rows: 1007913

Quantity:
Minimum: 1
Maximum: 80995
Non-positive: 0

Price:
Minimum: 0.001
Maximum: 25111.09
Non-positive: 0

Transaction Value:
Minimum: np.float64(0.001)
Non-positive: 0

Date range:
2009-12-01 07:45:00
2011-12-09 12:50:00

Missing values:
Invoice                  0
StockCode                0
Description              0
Quantity                 0
InvoiceDate              0
Price                    0
Customer ID         228488
Country                  0
SourceSheet              0
TransactionValue         0
dtype: int64


In [17]:
# Inspect the 15 transactions with the highest quantities

extreme_transactions = df_transactions.nlargest(
    15, "Quantity"
)

display(
    extreme_transactions[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "TransactionValue",
            "InvoiceDate",
            "Country"
        ]
    ]
)

,Invoice,StockCode,Description,Quantity,Price,TransactionValue,InvoiceDate,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,2011-12-09 09:15:00,United Kingdom
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,2011-01-18 10:01:00,United Kingdom
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.20,2010-02-15 11:57:00,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.00,2010-03-17 13:09:00,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.00,2010-03-17 13:09:00,Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.40,2010-03-17 13:09:00,Denmark
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.00,2010-03-17 13:09:00,Denmark
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,United Kingdom
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,United Kingdom
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,United Kingdom


In [18]:
print("Missing descriptions:",
      df_transactions["Description"].isna().sum())

print("Missing Customer IDs:",
      df_transactions["Customer ID"].isna().sum())

Missing descriptions: 0
Missing Customer IDs: 228488


In [19]:
print("MISSING DESCRIPTION DISTRIBUTION")
print("-" * 45)

print(
    "In product dataset:",
    df_product["Description"].isna().sum()
)

print(
    "In positive-price dataset:",
    df_valid_price["Description"].isna().sum()
)

print(
    "In zero-price dataset:",
    df_zero_price["Description"].isna().sum()
)

print(
    "In negative-price dataset:",
    df_negative_price["Description"].isna().sum()
)

print("\nMISSING CUSTOMER ID DISTRIBUTION")

print(
    "In purchase dataset:",
    df_purchases["Customer ID"].isna().sum()
)

print(
    "In positive-price dataset:",
    df_valid_price["Customer ID"].isna().sum()
)

MISSING DESCRIPTION DISTRIBUTION
---------------------------------------------
In product dataset: 1642
In positive-price dataset: 0
In zero-price dataset: 1642
In negative-price dataset: 0

MISSING CUSTOMER ID DISTRIBUTION
In purchase dataset: 231044
In positive-price dataset: 228488


In [20]:
Q1 = df_transactions["Quantity"].quantile(0.25)
Q3 = df_transactions["Quantity"].quantile(0.75)

IQR = Q3 - Q1

upper_bound = Q3 + (3 * IQR)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Extreme quantity threshold:", upper_bound)

# Flag, do not delete
df_transactions["ExtremeQuantityFlag"] = (
    df_transactions["Quantity"] > upper_bound
)

print("\nFlagged records:",
      df_transactions["ExtremeQuantityFlag"].sum())

display(
    df_transactions[df_transactions["ExtremeQuantityFlag"]]
    .nlargest(15, "Quantity")[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "TransactionValue"]
    ]
)

Q1: 1.0
Q3: 12.0
IQR: 11.0
Extreme quantity threshold: 45.0

Flagged records: 37865


,Invoice,StockCode,Description,Quantity,Price,TransactionValue
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.20
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.00
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.00
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.40
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.00
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2500.00
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2500.00
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2500.00


In [21]:
from pathlib import Path

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "cleaned_transactions": df_transactions,
    "purchases": df_purchases,
    "cancellations": df_cancellations,
    "returns": df_returns,
    "zero_quantity": df_zero_quantity,
    "zero_price": df_zero_price,
    "negative_price": df_negative_price
}

for name, dataset in datasets.items():
    output_path = PROCESSED_DIR / f"{name}.csv"
    dataset.to_csv(output_path, index=False)
    print(f"Saved {name}: {len(dataset):,} rows")

print("\nAll datasets saved successfully!")

Saved cleaned_transactions: 1,007,913 rows
Saved purchases: 1,010,539 rows
Saved cancellations: 19,104 rows
Saved returns: 3,393 rows
Saved zero_quantity: 0 rows
Saved zero_price: 2,621 rows
Saved negative_price: 5 rows

All datasets saved successfully!
